# QGAIN v4.1.0 — Standardized validation and figure completion

This notebook creates a **figure-only scientific supplement** from the immutable `qgain-v4.1.0` freeze. It does not decode audio, recompute feature values, modify the measurement freeze, construct a family scalar, or calibrate an accept/reject threshold.

Required outputs: completed family evaluation workbook, machine-readable checklist, standardized panels A–H, optional ML handoff panel J, source data, captions, provenance, and a candidate manifest ready for atomic sealing.

In [1]:
from __future__ import annotations

import json
import shutil
import sys
from pathlib import Path

import pandas as pd


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'MAIN outputs reviewed').exists() and (candidate / 'src reviewed').exists():
            return candidate
    raise RuntimeError('Could not locate the reviewed pipeline project root.')

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
sys.path.insert(0, str(PROJECT_ROOT / 'src reviewed'))

from paper1_qc_reviewed.qgain_figure_completion_v100 import build_package

FREEZE_ROOT = PROJECT_ROOT / 'MAIN outputs reviewed' / '06_family_freezes' / 'gain_dynamics' / 'qgain-v4.1.0'
OUTPUT_ROOT = PROJECT_ROOT / 'outputs reviewed' / 'gain_dynamics' / 'qgain-v4.1.0-figures-v1.0.0-candidate'
DOCS_ROOT = PROJECT_ROOT / 'docs reviewed'

print({'project_root': str(PROJECT_ROOT), 'freeze_root': str(FREEZE_ROOT), 'output_root': str(OUTPUT_ROOT)})

{'project_root': 'C:\\Users\\musikicn\\Desktop\\Nevena_project\\Paper_1\\paper_1', 'freeze_root': 'C:\\Users\\musikicn\\Desktop\\Nevena_project\\Paper_1\\paper_1\\MAIN outputs reviewed\\06_family_freezes\\gain_dynamics\\qgain-v4.1.0', 'output_root': 'C:\\Users\\musikicn\\Desktop\\Nevena_project\\Paper_1\\paper_1\\outputs reviewed\\gain_dynamics\\qgain-v4.1.0-figures-v1.0.0-candidate'}


In [2]:
freeze_manifest = json.loads((FREEZE_ROOT / 'manifests' / 'qgain_v410_freeze_manifest.json').read_text(encoding='utf-8'))
assert freeze_manifest['measurement_version'] == 'qgain-v4.1.0'
assert freeze_manifest['freeze_status'] == 'frozen'
assert freeze_manifest['recording_count'] == 519
assert freeze_manifest['participant_count'] == 224
print('Validated immutable source freeze.')
display(pd.DataFrame([freeze_manifest]).T.rename(columns={0: 'value'}).head(25))

Validated immutable source freeze.


,value
measurement_version,qgain-v4.1.0
family,QGAIN
family_display_name,Recorded level and level dynamics
freeze_status,frozen
candidate_only,False
freeze_allowed,True
source_measurement_version,qgain-v4.0.1-candidate
source_cohort_extraction_completed,True
source_artifact_inventory_sha256,3fa34d913a6106d0984b1edbd7116d551b98354edd6229...
numerical_equivalence_to_v401,True


In [3]:
build_package(FREEZE_ROOT, OUTPUT_ROOT)

(OUTPUT_ROOT / 'docs').mkdir(parents=True, exist_ok=True)
(OUTPUT_ROOT / 'tables').mkdir(parents=True, exist_ok=True)

for name in [
    'QGAIN_Family_Evaluation_Workbook_v1_0.docx',
    'QGAIN_Standardized_Validation_and_Figure_Package_README.md',
    'QGAIN_FIGURE_COMPLETION_CONTRACT_v1_0.md',
]:
    shutil.copy2(DOCS_ROOT / name, OUTPUT_ROOT / 'docs' / name)

for name in [
    'QGAIN_Validation_Checklist_v1_0.csv',
    'QGAIN_Ten_Domain_Dashboard_v1_0.csv',
    'QGAIN_Figure_Gallery_Index_v1_0.csv',
]:
    shutil.copy2(DOCS_ROOT / name, OUTPUT_ROOT / 'tables' / name)

print('Generated standardized QGAIN figure package and attached completed workbook/checklists.')

Generated standardized QGAIN figure package and attached completed workbook/checklists.


In [4]:
figure_index = pd.read_csv(OUTPUT_ROOT / 'tables' / 'qgain_v410_figure_gallery_index.csv')
checklist = pd.read_csv(OUTPUT_ROOT / 'tables' / 'QGAIN_Validation_Checklist_v1_0.csv')
domain = pd.read_csv(OUTPUT_ROOT / 'tables' / 'QGAIN_Ten_Domain_Dashboard_v1_0.csv')
manifest = json.loads((OUTPUT_ROOT / 'manifests' / 'qgain_v410_figure_package_manifest.json').read_text(encoding='utf-8'))

assert len(figure_index) == 32
assert set('ABCDEFGH').issubset(set(figure_index['panel']))
assert figure_index['status'].eq('PASS').all()
assert manifest['feature_values_recomputed'] is False
assert manifest['required_panels_complete'] is True

display(domain)
display(figure_index.groupby(['panel', 'status']).size().rename('figure_count').reset_index())
display(pd.DataFrame([manifest]).T.rename(columns={0: 'value'}))

,#,Domain,Evaluation question,Gate mapping,Status,Evidence,Conclusion
0,1,Construct validity,Does the observable logically correspond to th...,"G1, G5, G10",PASS,feature registry; freeze contract; B01,The redefined construct is recorded operating ...
1,2,Estimator validity,Does the mathematical definition measure the i...,"G1, G2, G4",PASS,feature passports; G2 checks; A01-A03,"Median AC-RMS level, within-segment IQR, betwe..."
2,3,Implementation validity,Does the code implement the definition exactly...,"G1, G2",PASS,executed notebook; qgain_v410.py; 22 tests; fr...,The final values are bitwise equivalent to the...
3,4,Transformation behavior,Are preregistered invariance/equivariance prop...,G3,PASS,C01-C04; source G3 tables,Typical level is 1:1 gain-equivariant; dynamic...
4,5,Dose response,Does the estimator respond monotonically and q...,G4,PASS,A01-A03; source G4 tables,All intended estimators show ordered construct...
5,6,Discriminant validity,Does the feature avoid or explicitly bound res...,G5,CONDITIONAL,B01-B02; G5 checks; registry claim limits,Spectral redistribution at fixed RMS does not ...
6,7,Support and uncertainty,"Are support, availability, missingness, uncert...",G6,PASS,D01-D04; recording table; drift CI fields,"All values travel with availability, status, s..."
7,8,Reliability and robustness,Are values stable under reasonable perturbatio...,"G6, G8",CONDITIONAL,E01-E03; H01-H03,Typical level and within-segment IQR are robus...
8,9,Interpretability,"Are units, orientation, empirical range, examp...","G7, G10",PASS,F01-F04; G01-G08; feature passports,"All features have explicit units, nonordinal i..."
9,10,Scientific scope,Is evidential maturity and permitted claim sta...,"G1, G10",PASS,registry; feature passports; freeze rationale,All four are study-specific robust descriptors...


,panel,status,figure_count
0,A,PASS,3
1,B,PASS,2
2,C,PASS,4
3,D,PASS,4
4,E,PASS,3
5,F,PASS,4
6,G,PASS,8
7,H,PASS,3
8,J,PASS,1


,value
family,QGAIN
family_display_name,Recorded level and level dynamics
measurement_version,qgain-v4.1.0
figure_package_version,qgain-v4.1.0-figures-v1.0.0
status,complete_candidate_ready_for_local_seal
source_freeze_manifest_sha256,d5beed4560760f5aa3da78776ec76edc51099b153db8f7...
source_freeze_inventory_sha256,6f6c579a37a5ca7e53ac429334b93d942f0ad8dc99c0b8...
source_executed_notebook_sha256,42e51c85d8f4d908319c51a0f7feb175d2f4efb9052233...
recording_count,519
participant_count,224


## Final local action

After this notebook finishes without errors:

1. Save the notebook.
2. Close JupyterLab.
3. Run `scripts reviewed/freeze_qgain_figure_package_v100.ps1`.

The seal script will rerun the package tests, preserve this executed notebook, generate a SHA-256 inventory, and atomically publish the immutable figure supplement under `MAIN outputs reviewed/07_figure_packages/gain_dynamics/qgain-v4.1.0-figures-v1.0.0`.